# `opera-rs` Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/opera-rs/blob/main/examples/opera_rs_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/opera-rs.svg)](https://pypi.org/project/opera-rs/)
[![crates.io](https://img.shields.io/crates/v/opera-rs.svg)](https://crates.io/crates/opera-rs)
[![Docs](https://img.shields.io/badge/docs-opera--rs.opengeos.org-teal.svg)](https://opera-rs.opengeos.org)

**`opera-rs`** is a pure-Rust, agent-centric toolkit for searching, visualizing, and analyzing [NASA OPERA](https://www.jpl.nasa.gov/go/opera) satellite products (DSWx-HLS, DSWx-S1, DIST-ALERT, DIST-ANN, RTC-S1, CSLC-S1, DISP-S1) via NASA's public CMR API, [`titiler-cmr`](https://github.com/developmentseed/titiler-cmr), and [`terrio`](https://github.com/opengeos/terrio), with **zero GDAL dependencies** and **no Earthdata login required**.

This notebook demonstrates:
1. Exploring the 9 supported NASA OPERA products and band-math expression presets
2. Geocoding a place name and searching NASA CMR granules (with GeoPackage export via `terrio`)
3. Point sampling and 30 m zonal categorical surface-water statistics (`point`, `stats`)
4. Two-date surface-water change detection over Tulare Lake, California (`detect_change`)
5. Multi-date time-series trend analysis (`analyze_time_series`)
6. Observed flood-extent polygon vectorization (`derive_flood_extent`)
7. Pure-Rust PNG map preview rendering and offline `titiler-cmr` URL generation (`preview`, `tile_urls`)
8. Running the bundled `opera` CLI and inspecting its `--json-usage` schema

In [ ]:
%pip install -q opera-rs matplotlib pandas pillow

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import opera_rs
import pandas as pd
from PIL import Image

out_dir = Path("opera_output")
out_dir.mkdir(exist_ok=True)
print(f"opera-rs version: {opera_rs.__version__}")

## 1. Explore the NASA OPERA Product Catalog

In [ ]:
catalog_df = pd.DataFrame(
    [
        {
            "short_name": p["short_name"],
            "short_title": p["short_title"],
            "backend": p["backend"],
            "default_bands": ", ".join(p["default_bands"]),
            "description": p["description"],
        }
        for p in opera_rs.products()
    ]
)
catalog_df

In [ ]:
dswx = opera_rs.find_product("dswx-hls")
print("Resolved alias 'dswx-hls' ->", dswx["short_name"], "| default bands:", dswx["default_bands"])

rtc_presets = opera_rs.expression_presets("rtc-s1", "VV")
print("RTC-S1 VV expression presets:")
for preset in rtc_presets:
    print(f"  - {preset['label']}: {preset['expression']} (rescale={preset.get('rescale')})")

## 2. Geocode a Place and Search NASA CMR Granules

We use Tulare Lake in California's Central Valley during the historic spring 2023 re-emergence. Footprints can be written directly to GeoJSON, GeoPackage (`.gpkg`), or Shapefile (`.shp`) via `terrio` with zero GDAL dependencies.

In [ ]:
place_info = opera_rs.geocode("Tulare Lake, California")
print("Geocoded:", place_info["display_name"])
print("Nominatim bbox:", place_info["bbox"])

# Focused WGS 84 bounding box [west, south, east, north] over Tulare Lake
bbox = [-119.95, 35.85, -119.45, 36.25]

search_res = opera_rs.search(
    "dswx-hls",
    bbox=bbox,
    start="2023-04-20",
    end="2023-04-30",
    limit=10,
)
print(
    f"Found {search_res['hits']} total hits across {len(search_res['dates'])} date(s); "
    f"returned {len(search_res['granules'])} granule(s)."
)

gpkg_path = out_dir / "tulare_footprints.gpkg"
written = opera_rs.write_vector(
    search_res["feature_collection"],
    gpkg_path,
    layer_name="footprints",
)
print(f"Wrote {written} footprint polygon(s) to {gpkg_path}")
print("Read AOI bbox back from GeoPackage:", opera_rs.read_aoi_bbox(gpkg_path))

pd.DataFrame(
    [
        {
            "id": g["id"],
            "begin_date": g["begin_date"],
            "tile": g.get("tile"),
            "bands": ", ".join(g.get("bands", [])[:4]),
        }
        for g in search_res["granules"][:5]
    ]
)

## 3. Point Sampling and Zonal Categorical Water Statistics

`opera_rs.point()` samples pixel values at `(lon, lat)` with automatic categorical class-label decoding. `opera_rs.stats()` computes 30 m class pixel counts, percentages, and surface area in km^2 over the AOI.

In [ ]:
pt = opera_rs.point(
    -119.72,
    36.05,
    product="dswx-hls",
    temporal="2023-04-20/2023-04-30",
)
print("Point sample at (-119.72, 36.05):")
for asset in pt["assets"]:
    print(" ", asset["name"], "->", asset["values"], asset["class_labels"])

st = opera_rs.stats(
    bbox,
    product="dswx-hls",
    start="2023-04-20",
    end="2023-04-30",
)
first_band = next(iter(st["bands"].values()))
class_df = pd.DataFrame(first_band["class_breakdown"])
display(class_df)

fig, ax = plt.subplots(figsize=(8, 3.8))
plot_df = class_df.sort_values("area_km2", ascending=True)
ax.barh(plot_df["label"], plot_df["area_km2"], color="#0077b6")
ax.set_xlabel("Area (km^2)")
ax.set_title("Tulare Lake DSWx-HLS Surface Water Breakdown (Apr 20-30, 2023)")
plt.tight_layout()
plt.show()

## 4. Two-Date Surface Water Change Detection

Compare dry winter conditions (January 2023) against peak spring flooding (April 2023) with `opera_rs.detect_change()`.

In [ ]:
chg = opera_rs.detect_change(
    bbox,
    "2023-01-15",
    "2023-04-30",
    product="dswx-hls",
)
print(f"Before scene: {chg['before']['acquired_date']} ({chg['before']['granule_id']})")
print(f"After scene:  {chg['after']['acquired_date']} ({chg['after']['granule_id']})")
print(f"Open water delta:  {chg['open_water_delta_km2']:+.2f} km^2")
print(f"Total water delta: {chg['total_water_delta_km2']:+.2f} km^2")
pd.DataFrame(chg["class_deltas"])

## 5. Multi-Date Time-Series Trend Analysis

In [ ]:
ts = opera_rs.analyze_time_series(
    bbox,
    "2023-01-01",
    "2023-05-15",
    product="dswx-hls",
    interval_days=20,
    max_observations=6,
)
ts_df = pd.DataFrame(
    [
        {
            "date": obs["acquired_date"],
            "open_water_km2": obs.get("open_water_km2"),
            "total_water_km2": obs.get("total_water_km2"),
            "valid_percent": obs["stats"].get("valid_percent"),
        }
        for obs in ts["observations"]
    ]
)
print("Trend summary:", ts.get("trend"))
display(ts_df)

if not ts_df.empty:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(ts_df["date"], ts_df["open_water_km2"], marker="o", label="Open water (km^2)")
    ax.plot(ts_df["date"], ts_df["total_water_km2"], marker="s", linestyle="--", label="Total water (km^2)")
    ax.set_ylabel("Surface Water Area (km^2)")
    ax.set_title("Tulare Lake DSWx-HLS Surface Water Time Series (Jan-May 2023)")
    ax.legend()
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

## 6. Observed Flood-Extent Polygon Vectorization

`opera_rs.derive_flood_extent()` unions DSWx-HLS and cloud-penetrating DSWx-S1 water-only tiles, stitches them into a binary mask, traces exterior rings with dry-land holes, and returns an RFC 7946 GeoJSON `FeatureCollection`.

In [ ]:
flood = opera_rs.derive_flood_extent(
    bbox,
    "2023-04-20",
    "2023-04-30",
    max_granules=3,
    max_tiles_across=3,
)
print(
    f"Vectorized {flood['polygon_count']} polygon(s) "
    f"covering {flood['total_area_km2']:.2f} km^2 "
    f"(products: {', '.join(flood['products_used'])})"
)

flood_geojson = out_dir / "tulare_flood.geojson"
opera_rs.write_vector(flood["feature_collection"], flood_geojson)
print(f"Saved flood polygons to {flood_geojson}")

fig, ax = plt.subplots(figsize=(6.5, 6))
for feat in flood["feature_collection"]["features"]:
    geom = feat["geometry"]
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poly in polys:
        ring = poly[0]
        xs, ys = zip(*ring)
        ax.fill(xs, ys, facecolor="#0077b6", edgecolor="#023e8a", alpha=0.65, linewidth=0.5)
ax.set_xlim(bbox[0], bbox[2])
ax.set_ylim(bbox[1], bbox[3])
ax.set_aspect("equal")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("OPERA Observed Surface-Water Polygons (Tulare Lake, Apr 2023)")
plt.tight_layout()
plt.show()

## 7. Pure-Rust Map Preview and Offline `titiler-cmr` URLs

In [ ]:
preview_path = out_dir / "tulare_preview.png"
png_bytes = opera_rs.preview(
    bbox,
    product="dswx-hls",
    start="2023-04-20",
    end="2023-04-30",
    water_only=False,
    basemap=False,
    max_dim=768,
    title="Tulare Lake DSWx-HLS (Apr 2023)",
    output_path=preview_path,
)
print(f"Rendered {len(png_bytes):,} bytes to {preview_path}")

img = Image.open(io.BytesIO(png_bytes))
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()

# Offline tile URL templates (rasterio and xarray backends)
dswx_urls = opera_rs.tile_urls(
    search_res["concept_id"],
    product="dswx-hls",
    water_only=True,
)
disp_urls = opera_rs.tile_urls(
    "C3294057315-ASF",
    product="disp-s1",
    backend="xarray",
    variables=["displacement"],
    group="/",
)
print("DSWx-HLS TileJSON URL:", dswx_urls["tilejson_url"])
print("DISP-S1 TileJSON URL: ", disp_urls["tilejson_url"])

## 8. Bundled `opera` CLI and Agent `--json-usage` Schema

The `opera-rs` Python package bundles both the `opera` CLI and the `opera mcp` server.

In [ ]:
import json
import subprocess
import sys

ver = subprocess.run([sys.executable, "-m", "opera_rs", "--version"], capture_output=True, text=True, check=True)
print("Bundled CLI version:", ver.stdout.strip())

usage = subprocess.run([sys.executable, "-m", "opera_rs", "--json-usage"], capture_output=True, text=True, check=True)
schema = json.loads(usage.stdout)
print("Schema program:", schema["program"], "| version:", schema["version"])
print("Subcommands:", [c["name"] for c in schema["command"]["subcommands"]])